In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [2]:
file_path = "../data/wine+quality/winequality-white.csv"

In [3]:
data = pd.read_csv(file_path,sep=';')

In [4]:
df = pd.DataFrame(data)

In [5]:
df.dtypes

fixed acidity           float64
volatile acidity        float64
citric acid             float64
residual sugar          float64
chlorides               float64
free sulfur dioxide     float64
total sulfur dioxide    float64
density                 float64
pH                      float64
sulphates               float64
alcohol                 float64
quality                   int64
dtype: object

In [6]:
df.var(numeric_only=True)

fixed acidity              0.712114
volatile acidity           0.010160
citric acid                0.014646
residual sugar            25.725770
chlorides                  0.000477
free sulfur dioxide      289.242720
total sulfur dioxide    1806.085491
density                    0.000009
pH                         0.022801
sulphates                  0.013025
alcohol                    1.514427
quality                    0.784356
dtype: float64

## Zadanie 1

In [7]:
values_max_var = sorted(df.var(numeric_only = True),)[-4:]

In [8]:
values_max_var

[1.5144269817873437, 25.725770164385946, 289.2427199993196, 1806.0854908480976]

## Zadanie 2

In [9]:
def detect_outliers(df):
    outlier_columns = []

    numeric_df = df.select_dtypes(include=['number'])

    for column in numeric_df.columns:
        Q1 = numeric_df[column].quantile(0.25)
        Q3 = numeric_df[column].quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        has_outliers = ((numeric_df[column] < lower_bound) |
                        (numeric_df[column] > upper_bound)).any()

        if has_outliers:
            outlier_columns.append(column)

    return outlier_columns

## Zadanie 3

In [10]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [13]:
knn_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_unscaled.fit(X_train, y_train)
y_pred_unscaled = knn_unscaled.predict(X_test)

In [14]:
knn_std = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])
knn_std.fit(X_train, y_train)
y_pred_std = knn_std.predict(X_test)

In [15]:
knn_minmax = Pipeline([
    ('scaler', MinMaxScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])
knn_minmax.fit(X_train, y_train)
y_pred_minmax = knn_minmax.predict(X_test)

In [16]:
knn_robust = Pipeline([
    ('scaler', RobustScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])
knn_robust.fit(X_train, y_train)
y_pred_robust = knn_robust.predict(X_test)

In [17]:
print(f"Dokładność k-NN bez skalowania: {accuracy_score(y_test, y_pred_unscaled):.4f}")
print(f"Dokładność k-NN ze skalowaniem standardowym: {accuracy_score(y_test, y_pred_std):.4f}")
print(f"Dokładność k-NN ze skalowaniem min-max: {accuracy_score(y_test, y_pred_minmax):.4f}")
print(f"Dokładność k-NN ze skalowaniem metodą robust: {accuracy_score(y_test, y_pred_robust):.4f}")

Dokładność k-NN bez skalowania: 0.9591
Dokładność k-NN ze skalowaniem standardowym: 0.9591
Dokładność k-NN ze skalowaniem min-max: 0.9649
Dokładność k-NN ze skalowaniem metodą robust: 0.9649


## Zadanie 4

In [18]:
outlier_cols = detect_outliers(X)
print(f"Cechy z wartościami odstającymi ({len(outlier_cols)}/{len(X.columns)}):")
for col in outlier_cols:
    print(f"  - {col}")

non_outlier_cols = [c for c in X.columns if c not in outlier_cols]
print(f"\nCechy BEZ wartości odstających ({len(non_outlier_cols)}/{len(X.columns)}):")
for col in non_outlier_cols:
    print(f"  - {col}")

Cechy z wartościami odstającymi (29/30):
  - mean radius
  - mean texture
  - mean perimeter
  - mean area
  - mean smoothness
  - mean compactness
  - mean concavity
  - mean concave points
  - mean symmetry
  - mean fractal dimension
  - radius error
  - texture error
  - perimeter error
  - area error
  - smoothness error
  - compactness error
  - concavity error
  - concave points error
  - symmetry error
  - fractal dimension error
  - worst radius
  - worst texture
  - worst perimeter
  - worst area
  - worst smoothness
  - worst compactness
  - worst concavity
  - worst symmetry
  - worst fractal dimension

Cechy BEZ wartości odstających (1/30):
  - worst concave points


In [19]:
knn_std_outlier = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('scaler', StandardScaler(), outlier_cols)
    ], remainder='passthrough')),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])
knn_std_outlier.fit(X_train, y_train)
y_pred_std_outlier = knn_std_outlier.predict(X_test)

In [20]:
knn_minmax_outlier = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('scaler', MinMaxScaler(), outlier_cols)
    ], remainder='passthrough')),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])
knn_minmax_outlier.fit(X_train, y_train)
y_pred_minmax_outlier = knn_minmax_outlier.predict(X_test)

In [21]:
knn_robust_outlier = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('scaler', RobustScaler(), outlier_cols)
    ], remainder='passthrough')),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])
knn_robust_outlier.fit(X_train, y_train)
y_pred_robust_outlier = knn_robust_outlier.predict(X_test)

In [22]:
print("Porównanie wyników:")
print(f"{'Metoda':<45} {'Zad.3 (wszystkie)':<20} {'Zad.4'}")
print("-" * 85)
print(f"{'Bez skalowania':<45} {accuracy_score(y_test, y_pred_unscaled):<20.4f} {'-'}")
print(f"{'StandardScaler':<45} {accuracy_score(y_test, y_pred_std):<20.4f} {accuracy_score(y_test, y_pred_std_outlier):.4f}")
print(f"{'MinMaxScaler':<45} {accuracy_score(y_test, y_pred_minmax):<20.4f} {accuracy_score(y_test, y_pred_minmax_outlier):.4f}")
print(f"{'RobustScaler':<45} {accuracy_score(y_test, y_pred_robust):<20.4f} {accuracy_score(y_test, y_pred_robust_outlier):.4f}")

Porównanie wyników:
Metoda                                        Zad.3 (wszystkie)    Zad.4 (tylko outlier)
-------------------------------------------------------------------------------------
Bez skalowania                                0.9591               -
StandardScaler                                0.9591               0.9649
MinMaxScaler                                  0.9649               0.9708
RobustScaler                                  0.9649               0.9649


### Odpowiedzi

**Czy sam fakt występowania wartości odstających jest wskazówką do przeprowadzenia skalowania danych?**

Sama obecność outlierów nie jest jedynym powodem do skalowania, bo w naszym zbiorze mają je niemal wszystkie cechy (29/30). Kluczowym problemem są różne rzędy wielkości cech, które zaburzają działanie k-NN opartego na odległości. Selektywne skalowanie tylko wybranych kolumn dało tutaj wyniki porównywalne lub lepsze niż wrzucenie wszystkiego do jednego worka.

**Czy obecność wartości odstających może być czynnikiem decyzyjnym w kontekście zastosowania konkretnej metody skalowania danych?**

MinMaxScaler i StandardScaler słabo znoszą outliery, bo te mocno zniekształcają średnią lub sztucznie rozciągają skalę danych. W takich przypadkach najlepiej wrzucić RobustScaler, który bazuje na medianie i rozstępie międzykwartylowym. Dzięki temu model skupia się na głównej masie danych i nie wariuje przez pojedyncze, skrajne wartości.

**Jaki inny czynnik występujący w danych może kwalifikować cechę do poddania jej skalowaniu?**

Gdy cechy mają totalnie inne rozpiętości, np. tysiące kontra ułamki, algorytmy typu k-NN głupieją i widzą tylko te większe liczby. Duża wariancja jednej zmiennej potrafi zdominować resztę, przez co model ignoruje drobniejsze, ale ważne sygnały. Skalowanie wyrównuje szanse wszystkich cech, żeby żadna nie „przekrzykiwała” pozostałych samym rozmiarem.